In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/My Drive/Project/Project Data/filtered_features_dataset_2.csv')

Mounted at /content/drive


In [2]:
!pip install transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [23]:
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration
from transformers import AdamW
from tqdm import tqdm
import time


In [4]:
df = df.sample(frac=0.2, random_state=18)

In [5]:

# Clean the data by removing rows with missing values (or handle missing values)
df.dropna(inplace=True)

# Combine features into input string
df['input'] = df['pet'] + ' ' + df['age'] + ' ' + df['size'] + ' ' + df['gender'] + ' ' + df['color_code']
# Split into input (features) and target (descriptions)
inputs = df['input'].tolist()
targets = df['text'].tolist()

# Split data into training, validation, and test sets (80/10/10 split)
train_inputs, test_inputs, train_targets, test_targets = train_test_split(inputs, targets, test_size=0.1)
train_inputs, val_inputs, train_targets, val_targets = train_test_split(train_inputs, train_targets, test_size=0.1)


In [6]:
df.head()

,age,gender,size,pet,color_code,text,status,input
64298,Young,Female,Medium,Dog,Black,"This is Mocshie, she is a very sweet and gentl...",Adopted,Dog Young Medium Female Black
71971,Baby,Male,Large,Cat,Brown_Chocolate,Greetings! I'm Middleton. I was saved out of...,Adopted,Cat Baby Large Male Brown_Chocolate
11037,Adult,Female,Medium,Dog,White_Cream,"Hi, my name is Winter! I am a sweet 4-year-old...",Adopted,Dog Adult Medium Female White_Cream
17898,Adult,Male,Large,Dog,Black,Meet Zack! This playful young man joined us fr...,Adopted,Dog Adult Large Male Black
28345,Adult,Male,Extra Large,Dog,Brown_Chocolate,"Bosco is a 5 year old, male Chocolate Lab. He ...",Adopted,Dog Adult Extra Large Male Brown_Chocolate


In [7]:
# Load the Flan-T5 tokenizer
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-small')

# Tokenize input and target data
def preprocess_data(inputs, targets, max_length=512):
    input_encodings = tokenizer(inputs, max_length=max_length, padding=True, truncation=True, return_tensors="pt")
    target_encodings = tokenizer(targets, max_length=max_length, padding=True, truncation=True, return_tensors="pt")
    return input_encodings, target_encodings

# Preprocess training, validation, and test data
train_encodings, train_labels = preprocess_data(train_inputs, train_targets)
val_encodings, val_labels = preprocess_data(val_inputs, val_targets)
test_encodings, test_labels = preprocess_data(test_inputs, test_targets)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [8]:

class PetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Create datasets
train_dataset = PetDataset(train_encodings, train_labels)
val_dataset = PetDataset(val_encodings, val_labels)
test_dataset = PetDataset(test_encodings, test_labels)

# Create data loaders
train_loader = DataLoader(train_dataset,batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size= 8, shuffle=False)

In [9]:
# Load Flan-T5 model
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small')

# Move model to GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [10]:
#Set up the optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

#Training loop
for epoch in range(4):  # Example: 6 epochs
    model.train()  # Set model to training mode
    for batch in train_loader:
        # Move batch to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()  # Backpropagate

        optimizer.step()  # Update weights on GPU
        optimizer.zero_grad()  # Reset gradients

    print(f"Epoch {epoch + 1} completed.")

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
<ipython-input-8-22dd0698453e>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-8-22dd0698453e>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels['input_ids'][idx])
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.

Epoch 1 completed.
Epoch 2 completed.
Epoch 3 completed.
Epoch 4 completed.


In [11]:
model.eval()  # Set model to evaluation mode
eval_loss = 0
with torch.no_grad():  # Disable gradient computation for validation
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        eval_loss += outputs.loss.item()

avg_eval_loss = eval_loss / len(val_loader)
print(f"Validation Loss: {avg_eval_loss}")

<ipython-input-8-22dd0698453e>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-8-22dd0698453e>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels['input_ids'][idx])


Validation Loss: 0.8509182611649687


In [12]:
running_loss = 0
for batch in train_loader:
    # Move batch to GPU
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()  # Backpropagate

    optimizer.step()  # Update weights
    optimizer.zero_grad()  # Reset gradients

    running_loss += loss.item()

avg_train_loss = running_loss / len(train_loader)
print(f"Training Loss: {avg_train_loss}")

<ipython-input-8-22dd0698453e>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-8-22dd0698453e>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels['input_ids'][idx])


Training Loss: 0.7882575426234223


In [13]:
model.save_pretrained("flan_t5_trained_model")

In [14]:
def generate_description(pet_features, model, tokenizer, prompt_weight=1.5):
    input_text = (
    "Generate an engaging and lively pet adoption description. "
    "Highlight the pet's best traits and personality while making it irresistible for adoption. "
    "Use the following characteristics: " + " ".join(pet_features)
)
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

    # Generate the description with higher adherence to the prompt
    outputs = model.generate(input_ids,
                             max_length = 130,
                             num_beams=5,
                             early_stopping=True,
                             repetition_penalty = 1.8,
                             temperature=1.2,  # Lower randomness og: 0.7
                             length_penalty=1.2,
                             do_sample=True,
                             top_k=30,  # Use top-k sampling
                             top_p=0.9  # Use top-p (nucleus) sampling
                             )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)



In [15]:
# Example usage
new_pet = ['dog', 'brown', 'large', 'male']
print(generate_description(new_pet, model, tokenizer))


If you are interested in this pet and would like to meet it, please fill out an adoption application on our website.


In [16]:
pet1 = ['dog' , 'adult', 'tiny', 'male', 'white']
print(generate_description(pet1, model, tokenizer))


This is a very sweet and playful dog. He loves to play with other dogs and enjoys playing with them. He gets along well with other dogs and cats. He will be a great addition to any family. If you are interested in adopting this pet, please fill out an adoption application on our website. We look forward to meeting you!


In [17]:
pet2 = ['kelb',
        'ferħ',
        'zgħir',
      'maskil', 'iswed']
print(generate_description(pet2 , model,tokenizer))

This is a description of a pet that has been adopted from a shelter. This is a description of a pet that has been adopted from a shelter. This is a description of a pet that has been adopted from a shelter. This is an description of a pet that has been adopted from a shelter.


In [18]:
petempty = []

In [28]:


start_time = time.time()
description = generate_description(['parrot', 'adult', 'large', 'male', 'multicolored', 'talkative', 'social', 'intelligent', 'requires daily interaction', 'long lifespan'], model, tokenizer)
print(description)
latency = time.time() - start_time
print(f"Latency: {latency}s")


This is a list of all the adoptable pets that are available for adoption. You can fill out an adoption application online on our official website. All of the pet listings on our website are done by individuals and/or by staff only. You can also set up an appointment with a licensed pet shelter to view all of the pet listings on our website. For more information on how to set up an appointment, please visit our website at rhodecountyhumanesociety.org/adopt. All of the pet listings on our website are done by individuals and/or by staff only. You
Latency: 2.7121222019195557s


In [29]:
pet4 = ['','','','']
print(generate_description(pet4 , model,tokenizer))

You can fill out an adoption application online on our official website. If you are interested in learning more about the pet and would like to schedule a meeting with one of our volunteers, please email us at info@petshumanesociety.org.


In [30]:
petmalti = ['cat', 'kitten', '!!##', 'female', 'brown']
print(generate_description(petmalti , model,tokenizer))

This is a very sweet and playful girl. She loves to be with her people, but she doesn't like cats. She loves to play with toys and would do best in a home with a cat or kitten. If you are interested in adopting this pet, please fill out an adoption application on our website.


In [31]:
import unittest

class TestDataProcessing(unittest.TestCase):

    def test_preprocessing(self):
        # Test tokenization shapes for a sample batch
        sample_inputs = ["dog puppy small male brown"]
        sample_targets = ["A playful young brown puppy looking for a home."]
        input_encodings, target_encodings = preprocess_data(sample_inputs, sample_targets)
        self.assertLessEqual(input_encodings.input_ids.shape[1], 512, "Input encodings shape mismatch")
        self.assertLessEqual(target_encodings.input_ids.shape[1], 512, "Target encodings shape mismatch")

    def test_missing_values(self):
        # Ensure there are no null values in the dataframe after cleaning
        self.assertFalse(df.isnull().values.any(), "Dataframe contains null values after preprocessing")


class TestGenerateDescription(unittest.TestCase):
    def setUp(self):
        self.model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small').to(device)
        self.tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-small')

    def test_generate_output(self):
        sample_pet_features = ["dog", "puppy", "small", "male", "brown"]
        description = generate_description(sample_pet_features, self.model, self.tokenizer)

        # Ensure output is not empty
        self.assertTrue(len(description) > 0, "Generated description is empty")

    def test_repetition_penalty(self):
        sample_pet_features = ["hamster", "baby", "chunky", "male", "multicoloured"]
        description = generate_description(sample_pet_features, self.model, self.tokenizer)

        # Check for repetitive terms in the description
        words = description.split()
        repetition_count = sum(1 for i in range(len(words) - 1) if words[i] == words[i + 1])
        self.assertLess(repetition_count, 2, "Too much repetition in generated description")

    def test_rare_input(self):
        rare_pet_features = ["unicorn", "immortal", "giant", "male", "rainbow"]
        description = generate_description(rare_pet_features, self.model, self.tokenizer)

        # Ensure model generates something coherent for unknown input
        self.assertTrue(len(description) > 0, "Generated description for rare input is empty")



In [32]:
if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 4.219s

OK


In [33]:
import unittest

class TestPipeline(unittest.TestCase):
    def setUp(self):
        self.model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small').to(device)
        self.tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-small')

    def test_data_preprocessing(self):
        # Ensure no null values in the dataframe
        self.assertFalse(df.isnull().values.any(), "Dataframe contains null values after preprocessing")

        # Test tokenization shapes for a sample batch
        sample_inputs = ["dog puppy small male brown"]
        sample_targets = ["A playful young brown puppy looking for a home."]
        input_encodings, target_encodings = preprocess_data(sample_inputs, sample_targets)
        self.assertLessEqual(input_encodings.input_ids.shape[1], 512, "Input encodings shape mismatch")
        self.assertLessEqual(target_encodings.input_ids.shape[1], 512, "Target encodings shape mismatch")

    def test_description_generation(self):
        # Test valid output for regular input
        sample_pet_features = ["dog", "puppy", "small", "male", "brown"]
        description = generate_description(sample_pet_features, self.model, self.tokenizer)
        self.assertTrue(len(description) > 0, "Generated description is empty")

        # Test coherence and repetition penalty
        words = description.split()
        repetition_count = sum(1 for i in range(len(words) - 1) if words[i] == words[i + 1])
        self.assertLess(repetition_count, 2, "Too much repetition in generated description")

    def test_edge_cases(self):
        # Test handling of rare or edge case inputs
        rare_pet_features = ["unicorn", "immortal", "giant", "male", "rainbow"]
        description = generate_description(rare_pet_features, self.model, self.tokenizer)
        self.assertTrue(len(description) > 0, "Generated description for rare input is empty")

        # Test for empty input edge case
        empty_pet_features = []
        description = generate_description(empty_pet_features, self.model, self.tokenizer)
        self.assertTrue(len(description) > 0, "Generated description for empty input is empty")

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


........
----------------------------------------------------------------------
Ran 8 tests in 8.412s

OK


In [34]:
def test_empty_input():
    # Test with empty inputs
    inputs = [""]
    targets = [""]

    input_encodings, target_encodings = preprocess_data(inputs, targets)

    assert input_encodings['input_ids'].shape[1] == 1, "Tokenized empty input should have length 1."
    assert target_encodings['input_ids'].shape[1] == 1, "Tokenized empty target should have length 1."

    print("test_empty_input passed.")

In [35]:
def test_max_length_truncation():
    # Test with a long input (longer than max_length)
    inputs = ["This is a very long sentence that exceeds the max length." * 10]
    targets = ["This is the target sentence which also needs to be truncated." * 10]

    max_length = 20  # A small max length to ensure truncation

    input_encodings, target_encodings = preprocess_data(inputs, targets, max_length=max_length)

    assert input_encodings['input_ids'].shape[1] == max_length, "Input should be truncated to max_length."
    assert target_encodings['input_ids'].shape[1] == max_length, "Target should be truncated to max_length."

    print("test_max_length_truncation passed.")

In [36]:
if __name__ == "__main__":
    test_empty_input()
    test_max_length_truncation()

test_empty_input passed.
test_max_length_truncation passed.


In [20]:
# Specify the directory to save the model and tokenizer
output_dir = "./flan_t5_trained_model"

# Create the directory if it doesn't exist
import os
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the model and the tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model and tokenizer saved in {output_dir}")



Model and tokenizer saved in ./flan_t5_trained_model


In [21]:
from google.colab import files
import shutil

# Compress the folder into a ZIP file
shutil.make_archive('flan_t5_trained_model', 'zip', './flan_t5_trained_model')

# Download the ZIP file
files.download('flan_t5_trained_model.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
pet66 = ['cat' , 'baby', 'small', 'female', 'white']
print(generate_description(pet66,model ,tokenizer))


This is a very sweet little girl. She loves to play with toys and other animals. She would love to be your new best friend. If you are interested in adopting her, please fill out an adoption application on our website.
